In [1]:
Board = [['a1','a2','a3','a4','a5','a6','a7','a8'],
         ['b1','b2','b3','b4','b5','b6','b7','b8'],
         ['c1','c2','c3','c4','c5','c6','c7','c8'],
         ['d1','d2','d3','d4','d5','d6','d7','d8'],
         ['e1','e2','e3','e4','e5','e6','e7','e8'],
         ['f1','f2','f3','f4','f5','f6','f7','f8'],
         ['g1','g2','g3','g4','g5','g6','g7','g8'],
         ['h1','h2','h3','h4','h5','h6','h7','h8'],]

chess_imoj = [[],['♔','♕','♖','♗','♘','♙'],
              ['♚','♛','♜','♝','♞','♟']]

a = 0
b = 1
c = 2
d = 3
e = 4
f = 5
g = 6
h = 7

White = 1
Black =-1
File  = 0
Rank  = 1
Left  =-1
Right = 1
Up    = 1
Down  =-1
KingPos = [[],[e,1],[e,8]]

def PrintBoard(color):
    for i in range(25):
        print('-', end = '')
    print()
    scores = [0,0,0]
    for column in range(8):
        for row in range(8):
            if color == White:
                square = Board[7 - row][column]
            else:
                square = Board[row][7 - column]
            if type(square) != str:
                print("{:^1}".format(square.imoj), end = '')
                scores[square.color] += square.score
            elif square == '▒':
                print(square, end = '')
            else:
                square = list(square)
                square[0] = ord(square[0]) - 97
                square[1] = int(square[1])
                if (square[0] + square[1])%2 == 1:
                    print("{:^1}".format('■'), end = '')
                else:
                    print("{:^1}".format('□'), end = '')
        print()
    winning = int((scores[1] - scores[-1]) / abs(scores[1] - scores[-1])) if scores[1] - scores[-1] != 0 else 0
    if winning == 0:
        print('same score!')
    elif scores[1] < 100 or scores[-1] < 100:
        print(f'{chess_imoj[winning][0]} has won!')
        return winning
    else:
        print(f'{chess_imoj[winning][0]} + {abs(scores[1] - scores[-1])}')
    for i in range(25):
        print('-', end = '')
    print()
    return 0

def Boardcheck(x, y):  #해당 칸에 있는 기물의 색(혹은 0)을 반환.
    if Board[x][y - 1] == f'{chr(x+97)}{y}':
        return 0
    else:
        return Board[x][y - 1].color

def linecheck(cur_x, cur_y, dir_x, dir_y):  #양 끝을 제외한 한 줄(가로, 세로, 대각선)사이의 기물 갯수를 반환.
    result = 0
    way_x = sorted([cur_x, dir_x])
    way_x = list(range(way_x[0]+1,way_x[1]))
    way_y = sorted([cur_y, dir_y])
    way_y = list(range(way_y[0]+1,way_y[1]))

    if (dir_x-cur_x)*(dir_y-cur_y) < 0:
        way_x.reverse()
    if cur_x == dir_x:
        for i in way_y:
            result += abs(Boardcheck(cur_x, i))
    elif cur_y == dir_y:
        for i in way_x:
            result += abs(Boardcheck(i, cur_y))
    else:
        for i, j in zip(way_x,way_y):
            result += abs(Boardcheck(i, j))
    print(f'{result} pieces in way {(cur_x, cur_y), (dir_x, dir_y)}')
    return result

def safechecker(cur_x, cur_y, color):  # 해당 칸이 체크를 당하고 있는지 확인하는 함수
    directions = [
        (Right, 0),  # 오른쪽
        (Left, 0), # 왼쪽
        (0, Up),  # 아래
        (0, Down), # 위
        (Right, Up),  # 오른쪽 위 대각선
        (Right, Down), # 오른쪽 아래 대각선
        (Left, Up), # 왼쪽 위 대각선
        (Left, Down) # 왼쪽 아래 대각선
    ]
    for dx, dy in directions:
        for i in range(1, 8):
            new_x, new_y = cur_x + dx * i, cur_y + dy * i
            # 보드 경계 체크
            if not (0 <= new_x < 8 and 1 <= new_y < 9):
                break
            piece = Boardcheck(new_x, new_y)

            if piece == -color:  # 상대방의 피스
                if dx * dy == 0 and 100 > Board[new_x][new_y-1].score >= 5:  # 점수가 5 이상인 경우(룩 혹은 퀸)
                    print(f'{chr(new_x + 97)}{new_y}')
                    return 'CHECK'
                elif abs(dx * dy) == 1 and Board[new_x][new_y-1].score % 3 == 0 and not "Knight" in Board[new_x][new_y-1].name:
                    print(f'{chr(new_x + 97)}{new_y}')
                    return 'CHECK'
                break  # 같은 색의 피스가 있으면 종료
            elif piece == color:  # 같은 색의 피스
                break
    knight_check = [
        (cur_x + 2, cur_y + 1), (cur_x + 1, cur_y + 2),
        (cur_x - 2, cur_y + 1), (cur_x - 1, cur_y + 2),
        (cur_x - 2, cur_y - 1), (cur_x - 1, cur_y - 2),
        (cur_x + 2, cur_y - 1), (cur_x + 1, cur_y - 2)
    ]
    valid_knights = []
    for i in knight_check:
        if 0 <= i[0] < 8 and 1 <= i[1] < 9:
            valid_knights.append(i)
    for i in valid_knights:
        if Boardcheck(i[0], i[1]) == -color and "Knight" in Board[i[0]][i[1] - 1].name:
            return "CHECK"
    return 'SAFE'


class Piece:
    def __init__(self, name, color, x, y, imoj, score):
        self.name = name
        self.color = color
        self.cur = {"File":x,
                    "Rank":y}
        self.imoj = imoj
        self.score = score

    def __str__(self):
        return self.name

    def pos(self, x, y):
        Board[self.cur["File"]][self.cur["Rank"] - 1] = f'{chr(self.cur["File"]+97)}{self.cur["Rank"]}'
        self.cur["File"] = x
        self.cur["Rank"] = y
        Board[x][y - 1] = self

class Pawn(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 1)
        self.enpassent = 0

    def move(self, target_position):
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if self.cur["File"] == dir[0]:
            if dir[1] - self.cur["Rank"] == self.color:
                return self.move_1()
            elif dir[1] - self.cur["Rank"] == self.color * 2:
                return self.move_2()
            else:
                return "ERROR"
        elif dir[1] - self.cur["Rank"] == self.color:
            if self.cur["File"] - dir[0] == 1:
                return self.take(Left)
            elif self.cur["File"] - dir[0] == -1:
                return self.take(Right)
            else: return "ERROR"
        else:
            return "ERROR"

    def move_1(self):
        if Boardcheck(self.cur["File"], self.cur["Rank"]+self.color) == 0:
            self.pos(self.cur["File"],self.cur["Rank"]+self.color)
        else:
            return "ERROR"


    def move_2(self):
        if Boardcheck(self.cur["File"], self.cur["Rank"]+self.color) == 0 and\
         Boardcheck(self.cur["File"], self.cur["Rank"]+self.color+self.color) == 0 and\
         self.cur["Rank"] == 4.5 - self.color*2.5:
            self.pos(self.cur["File"],self.cur["Rank"]+self.color+self.color)
            if self.cur["File"] != 0:
                if Boardcheck(self.cur["File"] - 1, self.cur["Rank"]) == - self.color and \
                Board[self.cur["File"] - 1][self.cur["Rank"] - 1].score == 1:
                    Board[self.cur["File"] - 1][self.cur["Rank"] - 1].enpassent = 1
            if self.cur["File"] != 7:
                if Boardcheck(self.cur["File"] + 1, self.cur["Rank"]) == - self.color and \
                Board[self.cur["File"] + 1][self.cur["Rank"] - 1].score == 1:
                    Board[self.cur["File"] + 1][self.cur["Rank"] - 1].enpassent = -1
        else:
            return "ERROR"


    def take(self, dir):
        if self.cur["File"] != 3.5 + dir * 3.5 and self.cur["Rank"] != 4.5 + self.color*3.5:           #체스 판 범위 내에서만 움직이기
            if Boardcheck(self.cur["File"] + dir, self.cur["Rank"] +self.color) == - self.color:    #칸 대각선 위가 다른 색일 때
                self.pos(self.cur["File"] + dir,self.cur["Rank"] +self.color)
            elif self.enpassent == dir:
                self.pos(self.cur["File"] + dir, self.cur["Rank"])
                self.pos(self.cur["File"], self.cur["Rank"] +self.color)
            else:
                return "ERROR"
        else:
            return "ERROR"


    def promote(self):
        print(f"{self.name} is ready to promote!")
        new_piece = input(f"Promote {self.name} to(Queen, Rook, Knight, Bishop):")
        if new_piece == "Queen":
            Board[self.cur["File"]][self.cur["Rank"]-1] = Queen(f'Queen_{self.color}_(Promoted)', self.color, self.cur["File"], self.cur["Rank"]-1,chess_imoj[self.color][1])
        elif new_piece == "Rook":
            Board[self.cur["File"]][self.cur["Rank"]-1] = Rook(f'Rook_{self.color}_(Promoted)', self.color, self.cur["File"], self.cur["Rank"]-1,chess_imoj[self.color][2])
        elif new_piece == "Knight":
            Board[self.cur["File"]][self.cur["Rank"]-1] = Knight(f'Knight_{self.color}_(Promoted)', self.color, self.cur["File"], self.cur["Rank"]-1,chess_imoj[self.color][4])
        else:
            Board[self.cur["File"]][self.cur["Rank"]-1] = Bishop(f'Bishop_{self.color}_(Promoted)', self.color, self.cur["File"], self.cur["Rank"]-1,chess_imoj[self.color][3])

class Rook(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 5)
        self.moved = 0

    def move(self, target_position):
        self.moved = 1
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if (self.cur["File"] == dir[0] or  self.cur["Rank"] == dir[1]) and\
        linecheck(self.cur["File"], self.cur["Rank"], dir[0], dir[1]) == 0 and\
        abs(Boardcheck(dir[0], dir[1]) + self.color) != 2:
            self.pos(dir[0], dir[1])
        else:
            return "ERROR"


class Knight(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 3)

    def move(self, target_position):
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if (dir[0] - self.cur["File"])**2 + (dir[1] - self.cur["Rank"])**2 == 5 and\
        abs(Boardcheck(dir[0], dir[1]) + self.color) != 2 :                                 #나이트가 이동하려는 위치에 말이 있다면 색이 다른지 확인. 같으면 더했을 때 절댓값이 2가 됨.
            self.pos(dir[0], dir[1])
        else:
            return "ERROR"


class Bishop(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 3)

    def move(self, target_position):
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if abs(self.cur["File"] - dir[0]) == abs(self.cur["Rank"] - dir[1]) and\
        linecheck(self.cur["File"], self.cur["Rank"], dir[0], dir[1]) == 0 and\
        abs(Boardcheck(dir[0], dir[1]) + self.color) != 2:
            self.pos(dir[0], dir[1])
        else:
            return "ERROR"


class King(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 100)
        self.moved = 0

    def move(self, target_position):
        self.moved = 1
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if abs(self.cur["File"] - dir[0]) <= 1 and\
        abs(self.cur["Rank"] - dir[1]) <= 1 and\
        abs(Boardcheck(dir[0], dir[1]) + self.color) != 2:
            self.pos(dir[0], dir[1])
            KingPos[self.color] = [self.cur["File"], self.cur["Rank"]]
        else:
            return "ERROR"


    def castling(self, dir):
        if linecheck(self.cur["File"], self.cur["Rank"], int(3.5 + dir*3.5), self.cur["Rank"]) == 0 and\
        Board[int(3.5 + dir*3.5)][self.cur["Rank"]-1].moved == 0:
            i = 0
            while True:
                if self.cur["File"] + i == 8 or self.cur["File"] == -1:
                    break
                if safechecker(self.cur["File"] + i, self.cur["Rank"], self.color) == "CHECK":
                    return "ERROR"
                i += dir
            Board[int(3.5 + dir*3.5)][self.cur["Rank"]-1].pos(4 + dir* 1,self.cur["Rank"])
            self.pos(4 + dir* 2,self.cur["Rank"])
            KingPos[self.color] = [self.cur["File"], self.cur["Rank"]]
        else:
            print("Invalid castling")
            return "ERROR"


class Queen(Piece):
    def __init__(self, name, color, x, y, imoj):
        super().__init__(name, color, x, y, imoj, 9)

    def move(self, target_position):
        dir = list(target_position)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if (abs(self.cur["File"] - dir[0]) ==  abs(self.cur["Rank"] - dir[1]) or\
        (self.cur["File"] == dir[0] or  self.cur["Rank"] == dir[1])) and\
        linecheck(self.cur["File"], self.cur["Rank"], dir[0], dir[1]) == 0 and\
        abs(Boardcheck(dir[0], dir[1]) + self.color) != 2:
            self.pos(dir[0], dir[1])
        else:
            return "ERROR"


class Player:
    def __init__(self, name, color):
        self.name = name
        self.color = color

for i in range(8):
    Board[i][1] = Pawn(f'Pawn_White_{chr(i + 97)}',White, i, 2, chess_imoj[1][5])
    Board[i][6] = Pawn(f'Pawn_Black_{chr(i + 97)}',Black, i, 7, chess_imoj[2][5])

for color_1, i in {
    'White': White,
    'Black': Black
    }.items():
    Board[a][int(3.5 - i * 3.5)] = Rook(f'Rook_{color_1}_1',i, a, int(4.5 - i * 3.5),chess_imoj[i][2])
    Board[h][int(3.5 - i * 3.5)] = Rook(f'Rook_{color_1}_2',i, h, int(4.5 - i * 3.5),chess_imoj[i][2])
    Board[b][int(3.5 - i * 3.5)] = Knight(f'Knight_{color_1}_1',i, b, int(4.5 - i * 3.5),chess_imoj[i][4])
    Board[g][int(3.5 - i * 3.5)] = Knight(f'Knight_{color_1}_2',i, g, int(4.5 - i * 3.5),chess_imoj[i][4])
    Board[c][int(3.5 - i * 3.5)] = Bishop(f'Bishop_{color_1}_1',i, c, int(4.5 - i * 3.5),chess_imoj[i][3])
    Board[f][int(3.5 - i * 3.5)] = Bishop(f'Bishop_{color_1}_2',i, f, int(4.5 - i * 3.5),chess_imoj[i][3])
    Board[e][int(3.5 - i * 3.5)] = King(f'King_{color_1}',i, e, int(4.5 - i * 3.5),chess_imoj[i][0])
    Board[d][int(3.5 - i * 3.5)] = Queen(f'Queen_{color_1}',i, d, int(4.5 - i * 3.5),chess_imoj[i][1])

def turn(color):
    while True:
        selected = input("enter a square(ex: e2, O-O, O-O-O, resign):")
        if selected == 'O-O':
            if Board[KingPos[color][0]][KingPos[color][1]-1].castling(Right) == 'ERROR':
                continue
            break
        elif selected == 'O-O-O':
            if Board[KingPos[color][0]][KingPos[color][1]-1].castling(Left) == "ERROR":
                continue
            break
        elif selected == 'resign':
            Board[KingPos[color][0]][KingPos[color][1]-1] = '▒'
            return
        selected = list(selected)
        selected[0] = ord(selected[0]) - 97
        selected[1] = int(selected[1])
        if not (0 <= selected[0] <= 7 and\
        1 <= selected[1] <= 8):
            print("squre doesn't exist :(")
            continue
        elif Boardcheck(selected[0], selected[1]) != color:
            print("Your piece isn't there :(")
            continue
        dir = input("move to(ex: e4): ")
        pos = dir
        dir = list(dir)
        dir[0] = ord(dir[0]) - 97
        dir[1] = int(dir[1])
        if not (0 <= dir[0] <= 7 and\
        1 <= dir[1] <= 8):
            print("squre doesn't exist :(")
            continue
        taken = Board[dir[0]][dir[1]-1]
        if Board[selected[0]][selected[1]-1].move(pos) == 'ERROR':
            print("that move is invalid")
            continue
        print('checking for checks.....')
        if safechecker(KingPos[color][0], KingPos[color][1], color) == "CHECK":
            print("that piece can't move there.                    ")
            Board[selected[0]][selected[1]-1] = Board[dir[0]][dir[1]-1]
            Board[dir[0]][dir[1]-1] = taken
            continue

        for file in Board:
            for square in file:
                if type(square) != str and square.score == 1 and square.color == color:
                    square.enpassent = 0

        print()
        print(f'{Board[dir[0]][dir[1]-1].name}\'s curruent position: {chr( Board[dir[0]][dir[1]-1].cur["File"]+97)}{ Board[dir[0]][dir[1]-1].cur["Rank"]}')
        if safechecker(KingPos[-color][0], KingPos[-color][1], -color) == "CHECK":
            print("◇◆CHECK◇◆")
        break

i = White
PrintBoard(Black)
while True:
    turn(i)
    if abs(PrintBoard(i)) == 1:
        print("GOOD GAME!")
        break
    i = -i


-------------------------
♜♞♝♛♚♝♞♜
♟♟♟♟♟♟♟♟
□■□■□■□■
■□■□■□■□
□■□■□■□■
■□■□■□■□
♙♙♙♙♙♙♙♙
♖♘♗♕♔♗♘♖
same score!
-------------------------
checking for checks.....

Pawn_White_e's curruent position: e4
-------------------------
♖♘♗♔♕♗♘♖
♙♙♙□♙♙♙♙
□■□■□■□■
■□■♙■□■□
□■□■□■□■
■□■□■□■□
♟♟♟♟♟♟♟♟
♜♞♝♚♛♝♞♜
same score!
-------------------------
checking for checks.....

Pawn_Black_e's curruent position: e5
-------------------------
♜♞♝♛♚♝♞♜
♟♟♟♟■♟♟♟
□■□■□■□■
■□■□♟□■□
□■□■♙■□■
■□■□■□■□
♙♙♙♙□♙♙♙
♖♘♗♕♔♗♘♖
same score!
-------------------------
-------------------------
♖♘♗▒♕♗♘♖
♙♙♙□♙♙♙♙
□■□■□■□■
■□■♙■□■□
□■□♟□■□■
■□■□■□■□
♟♟♟■♟♟♟♟
♜♞♝♚♛♝♞♜
♚ has won!
GOOD GAME!


## 부족한 부분
<ul>
    <li>체크메이트 계산 없음</li>
    <li>스테일메이트 시 무조건 항복해야함: 무승부 유도 불가</li>
    <li>UI 구현되지 않음</li>
	<li>예외처리가 없음</li>
</ul>
